## Ingest qualifying directory json


### Step 0 - Parameters and Imports


In [0]:
%run "../utils/config"


In [0]:
dbutils.widgets.text("p_source_value", "")
v_source_value = dbutils.widgets.get("p_source_value")


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType, FloatType
from pyspark.sql.functions import col, lit, current_timestamp, to_timestamp, concat

### Step 1 - Read the json file


#### 1.1 Define the schema


In [0]:
qualifyings_schema = StructType(fields=[
  StructField("constructorId", IntegerType(), False),
  StructField("driverId", IntegerType(), True),
  StructField("number", IntegerType(), True),
  StructField("position", IntegerType(), True),
  StructField("q1", StringType(), True),
  StructField("q2", StringType(), True),
  StructField("q3", StringType(), True),
  StructField("qualifyId", IntegerType(), True),
  StructField("raceId", IntegerType(), True)
])


#### 1.2 Read the json file


In [ ]:
qualifying_df = (
    spark.read
        .format("json")
        .schema(qualifyings_schema)
        .option("multiline", True)
        .load(f"{raw_folder_path}/qualifying")
)


### Step 2 - Transform the data


In [0]:
final_qualifying_df = add_ingestion_date(
    qualifying_df
        .withColumnRenamed("qualifyId", "qualify_id")
        .withColumnRenamed("raceId", "race_id")
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumn("source", lit(v_source_value)))


### Step 3 - Write the output to parquet


In [ ]:
final_qualifying_df.write.format("delta").mode("overwrite").saveAsTable("f1_processed.qualifying")


In [0]:
dbutils.notebook.exit("success")